In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\chidv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Step 1: Data Loading

In [2]:

badminton = pd.read_csv(r"D:\DS\Projects\GEN_AI\TASK_2\Sentiment Analysis\reviews_badminton\data.csv")
tawa = pd.read_csv(r"D:\DS\Projects\GEN_AI\TASK_2\Sentiment Analysis\reviews_tawa\data.csv")
tea = pd.read_csv(r"D:\DS\Projects\GEN_AI\TASK_2\Sentiment Analysis\reviews_tea\data.csv")

In [3]:
tea.shape

(9170, 8)

In [4]:
tawa.columns

Index(['Reviewer_Name', 'Reviewer_Rating', 'Review_Title', 'Review_Text',
       'Place_of_Review', 'Date_of_Review', 'Up_Votes', 'Down_Votes'],
      dtype='object')

In [5]:
badminton.columns

Index(['Reviewer Name', 'Review Title', 'Place of Review', 'Up Votes',
       'Down Votes', 'Month', 'Review text', 'Ratings'],
      dtype='object')

In [6]:
tea.columns

Index(['reviewer_name', 'reviewer_rating', 'review_title', 'review_text',
       'place_of_review', 'Date_of_review', 'up_votes', 'Down_votes'],
      dtype='object')

## Step 2: Data Cleaning 

In [7]:
badminton.columns = badminton.columns.str.strip().str.lower().str.replace(" ", "_")

badminton.rename(columns={
    'ratings': 'reviewer_rating'
}, inplace=True)

In [8]:
tawa.columns = tawa.columns.str.strip().str.lower().str.replace(" ", "_")

In [9]:
tea.columns = tea.columns.str.strip().str.lower().str.replace(" ", "_")

In [10]:
print(badminton.columns)
print(tawa.columns)
print(tea.columns)

Index(['reviewer_name', 'review_title', 'place_of_review', 'up_votes',
       'down_votes', 'month', 'review_text', 'reviewer_rating'],
      dtype='object')
Index(['reviewer_name', 'reviewer_rating', 'review_title', 'review_text',
       'place_of_review', 'date_of_review', 'up_votes', 'down_votes'],
      dtype='object')
Index(['reviewer_name', 'reviewer_rating', 'review_title', 'review_text',
       'place_of_review', 'date_of_review', 'up_votes', 'down_votes'],
      dtype='object')


In [11]:
badminton['date_of_review'] = None
badminton.drop(columns=['month'], inplace=True)

In [12]:
columns_order = [
    'reviewer_name', 'reviewer_rating', 'review_title',
    'review_text', 'place_of_review', 'date_of_review',
    'up_votes', 'down_votes'
]

badminton = badminton[columns_order]
tawa = tawa[columns_order]
tea = tea[columns_order]

In [13]:
badminton.loc[:, 'product'] = 'badminton'
tawa.loc[:, 'product'] = 'tawa'
tea.loc[:, 'product'] = 'tea'

In [14]:
df = pd.concat([badminton, tawa, tea], ignore_index=True)

In [15]:
df

,reviewer_name,reviewer_rating,review_title,review_text,place_of_review,date_of_review,up_votes,down_votes,product
0,Kamal Suresh,4.0,Nice product,"Nice product, good quality, but price is now r...","Certified Buyer, Chirakkal",None,889.0,64.0,badminton
1,Flipkart Customer,1.0,Don't waste your money,They didn't supplied Yonex Mavis 350. Outside ...,"Certified Buyer, Hyderabad",None,109.0,6.0,badminton
2,A. S. Raja Srinivasan,1.0,Did not meet expectations,Worst product. Damaged shuttlecocks packed in ...,"Certified Buyer, Dharmapuri",None,42.0,3.0,badminton
3,Suresh Narayanasamy,3.0,Fair,"Quite O. K. , but nowadays the quality of the...","Certified Buyer, Chennai",None,25.0,1.0,badminton
4,ASHIK P A,1.0,Over priced,Over pricedJust â?¹620 ..from retailer.I didn'...,NaN,None,147.0,24.0,badminton
...,...,...,...,...,...,...,...,...,...
20214,Omm Prakash,5.0,Simply awesome,Nice for red tea.Valeu for moneyREAD MORE,"Certified Buyer, Dhamanagar",Omm Prakash,26.0,5.0,tea
20215,Ritu Raj,4.0,Good choice,niceREAD MORE,"Certified Buyer, Katihar District",Ritu Raj,19.0,4.0,tea
20216,Arun Saini,1.0,Terrible product,Tata Gold Vs Tata Tea Premium👍Tata Tea Premiu...,"Certified Buyer, Haridwar",Arun Saini,13.0,2.0,tea
20217,Amitabh Shahi,5.0,Just wow!,I believe that it's the best packaged tea in t...,"Certified Buyer, Darbhanga",Amitabh Shahi,32.0,10.0,tea


In [16]:
df.shape

(20219, 9)

In [17]:
df.isna().sum()

reviewer_name        10
reviewer_rating     246
review_title         10
review_text           8
place_of_review      50
date_of_review     8518
up_votes             10
down_votes           10
product               0
dtype: int64

In [18]:
df = df.drop(columns=['date_of_review'])

In [19]:
df['review_title'] = df['review_title'].fillna("")
df['place_of_review'] = df['place_of_review'].fillna("unknown")
df['reviewer_rating'] = df['reviewer_rating'].fillna(df['reviewer_rating'].mean())

In [20]:
df['review_text'] = df['review_text'].fillna("")

In [21]:
df.isna().sum()

reviewer_name      10
reviewer_rating     0
review_title        0
review_text         0
place_of_review     0
up_votes           10
down_votes         10
product             0
dtype: int64

In [22]:
df

,reviewer_name,reviewer_rating,review_title,review_text,place_of_review,up_votes,down_votes,product
0,Kamal Suresh,4.0,Nice product,"Nice product, good quality, but price is now r...","Certified Buyer, Chirakkal",889.0,64.0,badminton
1,Flipkart Customer,1.0,Don't waste your money,They didn't supplied Yonex Mavis 350. Outside ...,"Certified Buyer, Hyderabad",109.0,6.0,badminton
2,A. S. Raja Srinivasan,1.0,Did not meet expectations,Worst product. Damaged shuttlecocks packed in ...,"Certified Buyer, Dharmapuri",42.0,3.0,badminton
3,Suresh Narayanasamy,3.0,Fair,"Quite O. K. , but nowadays the quality of the...","Certified Buyer, Chennai",25.0,1.0,badminton
4,ASHIK P A,1.0,Over priced,Over pricedJust â?¹620 ..from retailer.I didn'...,unknown,147.0,24.0,badminton
...,...,...,...,...,...,...,...,...
20214,Omm Prakash,5.0,Simply awesome,Nice for red tea.Valeu for moneyREAD MORE,"Certified Buyer, Dhamanagar",26.0,5.0,tea
20215,Ritu Raj,4.0,Good choice,niceREAD MORE,"Certified Buyer, Katihar District",19.0,4.0,tea
20216,Arun Saini,1.0,Terrible product,Tata Gold Vs Tata Tea Premium👍Tata Tea Premiu...,"Certified Buyer, Haridwar",13.0,2.0,tea
20217,Amitabh Shahi,5.0,Just wow!,I believe that it's the best packaged tea in t...,"Certified Buyer, Darbhanga",32.0,10.0,tea


## Step 3: Text Preprocessing

## Starting ML flow

In [ ]:
import mlflow

mlflow.set_tracking_uri(
    "http://localhost:5000"
)
mlflow.openai.autolog()

In [29]:

mlflow.set_experiment("Sentiment Analysis")

2026/04/03 15:49:54 INFO mlflow.tracking.fluent: Experiment with name 'Sentiment Analysis' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1775211594849, experiment_id='1', last_update_time=1775211594849, lifecycle_stage='active', name='Sentiment Analysis', tags={}, workspace='default'>

In [27]:
def get_sentiment(rating):
    if rating >= 4:
        return 'positive'
    else:
        return 'negative'

df['sentiment'] = df['reviewer_rating'].apply(get_sentiment)

In [30]:
df[['reviewer_rating', 'sentiment']].head()

,reviewer_rating,sentiment
0,4.0,positive
1,1.0,negative
2,1.0,negative
3,3.0,negative
4,1.0,negative


In [31]:
def clean_text(text):
    if pd.isnull(text):   # handle NaN
        return ""
    
    text = re.sub(r'[^a-zA-Z]', ' ', str(text))
    text = text.lower()
    words = text.split()
    words = [w for w in words if w not in stopwords.words('english')]
    return " ".join(words)

df['clean_text'] = df['review_text'].apply(clean_text)

## Step 4: Feature Engineering

In [32]:

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_text'])

In [33]:
y = df['sentiment']

In [34]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 128917 stored elements and shape (20219, 3805)>

In [35]:
y

0        positive
1        negative
2        negative
3        negative
4        negative
           ...   
20214    positive
20215    positive
20216    negative
20217    positive
20218    positive
Name: sentiment, Length: 20219, dtype: object

### Train-Test Split

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [37]:
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(16175, 3805) (4044, 3805) (16175,) (4044,)


In [38]:
y_train = y_train.values
y_test = y_test.values

## Model building 

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

2026/04/03 16:09:19 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/04/03 16:09:21 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'd55a808

🏃 View run sincere-quail-301 at: http://localhost:5000/#/experiments/1/runs/d55a80802e9e4442b75bf2f80ebf9e94
🧪 View experiment at: http://localhost:5000/#/experiments/1


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [40]:
training_score= model.score(X_train, y_train)
training_score

0.9382998454404946

In [41]:
y_pred = model.predict(X_test)

## 5 Model Evaluation or test metrics 

In [42]:
from sklearn.metrics import classification_report, f1_score

print(classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

              precision    recall  f1-score   support

    negative       0.87      0.55      0.68       615
    positive       0.92      0.98      0.95      3429

    accuracy                           0.92      4044
   macro avg       0.90      0.77      0.82      4044
weighted avg       0.92      0.92      0.91      4044

F1 Score: 0.9117735428846051


### Model 2 -- Support vector Machine 

In [43]:
from sklearn import svm
clf = svm.SVC()

In [44]:
clf

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [45]:
clf.fit(X_train, y_train)

2026/04/03 16:13:58 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'a7d827b3927c4dbd9488a9563ef58b9f', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/03 16:14:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run dazzling-loon-915 at: http://localhost:5000/#/experiments/1/runs/a7d827b3927c4dbd9488a9563ef58b9f
🧪 View experiment at: http://localhost:5000/#/experiments/1


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [91]:
training_score=clf.score(X_train, y_train)
training_score

0.960370942812983

## There fore test score == 93% , train accuracy == 91% so model is `Generalising `

## Model 3 - naive_bayes

In [46]:
from sklearn.naive_bayes import GaussianNB

In [47]:
gnb = GaussianNB()

In [48]:
X_train=X_train.toarray()

In [49]:
gnb.fit(X_train, y_train)

2026/04/03 16:18:33 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'c4c54603845443d6a90fb9b5000187cb', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/03 16:18:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run smiling-whale-27 at: http://localhost:5000/#/experiments/1/runs/c4c54603845443d6a90fb9b5000187cb
🧪 View experiment at: http://localhost:5000/#/experiments/1


,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [50]:
training_score=gnb.score(X_train, y_train)
training_score

0.4903245749613601

In [51]:
y_pred = gnb.predict(X_test.toarray())

In [52]:
print(classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

              precision    recall  f1-score   support

    negative       0.20      0.89      0.33       615
    positive       0.95      0.37      0.54      3429

    accuracy                           0.45      4044
   macro avg       0.58      0.63      0.43      4044
weighted avg       0.84      0.45      0.50      4044

F1 Score: 0.5043405559984586


## There fore test score == 50% , train accuracy == 49% 

## Model 4 -- RandomForestClassifier

In [53]:
from  sklearn.ensemble import RandomForestClassifier
RC=RandomForestClassifier()
RC.fit(X_train, y_train)

2026/04/03 16:20:06 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '6a9a90092ca942bf883b2ef610719799', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/03 16:21:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run melodic-lamb-437 at: http://localhost:5000/#/experiments/1/runs/6a9a90092ca942bf883b2ef610719799
🧪 View experiment at: http://localhost:5000/#/experiments/1


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [54]:
RC.score(X_train, y_train)

0.9721792890262752

In [55]:
y_pred = RC.predict(X_test.toarray())
print(classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

              precision    recall  f1-score   support

    negative       0.81      0.60      0.69       615
    positive       0.93      0.97      0.95      3429

    accuracy                           0.92      4044
   macro avg       0.87      0.79      0.82      4044
weighted avg       0.91      0.92      0.91      4044

F1 Score: 0.9126293211608364


## There fore test score == 91% , train accuracy == 97% 

## Model 5 --  DTree

In [56]:
from sklearn import tree
dt = tree.DecisionTreeClassifier()
dt.fit(X_train, y_train)

2026/04/03 16:23:04 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f895a51701d748e4bda3a48ef21e1166', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


2026/04/03 16:23:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run salty-conch-311 at: http://localhost:5000/#/experiments/1/runs/f895a51701d748e4bda3a48ef21e1166
🧪 View experiment at: http://localhost:5000/#/experiments/1


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [57]:
dt.score(X_train, y_train)

0.9721792890262752

In [58]:
y_pred = dt.predict(X_test.toarray())

In [59]:
print(classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

              precision    recall  f1-score   support

    negative       0.71      0.69      0.70       615
    positive       0.94      0.95      0.95      3429

    accuracy                           0.91      4044
   macro avg       0.83      0.82      0.82      4044
weighted avg       0.91      0.91      0.91      4044

F1 Score: 0.9090044736032082


## Therefore test score == 90% , train accuracy == 97% 

## Model 6 -- K-Nearest Neighbors

In [60]:
from sklearn.neighbors import KNeighborsClassifier
knn=KNeighborsClassifier()
knn.fit(X_train, y_train)

2026/04/03 16:26:40 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '404aabdb282647a9b2906de5c3703454', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/03 16:26:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run gifted-elk-292 at: http://localhost:5000/#/experiments/1/runs/404aabdb282647a9b2906de5c3703454
🧪 View experiment at: http://localhost:5000/#/experiments/1


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [61]:
knn.score(X_train, y_train)

0.920370942812983

In [62]:
y_pred = knn.predict(X_test)
print(classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

              precision    recall  f1-score   support

    negative       0.88      0.44      0.59       615
    positive       0.91      0.99      0.95      3429

    accuracy                           0.91      4044
   macro avg       0.89      0.72      0.77      4044
weighted avg       0.90      0.91      0.89      4044

F1 Score: 0.8923821824793999


## There fore test score == 89% , train accuracy == 92% 

### WE DECLARE `LOGISTIC REGRESSION` WILL BE BEST FIT SENTEMENT ANALYSIS 

In [65]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    C=1.0,
    class_weight=None,
    dual=False,
    fit_intercept=True,
    intercept_scaling=1,
    l1_ratio=0.0,
    max_iter=100,
    n_jobs=None,
    penalty=None,
    random_state=None,
    solver='lbfgs',
    tol=0.0001,
    verbose=0,
    warm_start=False
)

model.fit(X_train, y_train)

2026/04/03 17:03:31 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '07e098c18dc34797b54e56b81d33f069', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
c:\Users\chidv\.conda\envs\batch405\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\chidv\.conda\envs\batch405\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the dat

🏃 View run powerful-skink-484 at: http://localhost:5000/#/experiments/1/runs/07e098c18dc34797b54e56b81d33f069
🧪 View experiment at: http://localhost:5000/#/experiments/1


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",None
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

In [66]:
knn.score(X_train, y_train)

0.920370942812983

In [67]:
y_pred = knn.predict(X_test)
print(classification_report(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

              precision    recall  f1-score   support

    negative       0.88      0.44      0.59       615
    positive       0.91      0.99      0.95      3429

    accuracy                           0.91      4044
   macro avg       0.89      0.72      0.77      4044
weighted avg       0.90      0.91      0.89      4044

F1 Score: 0.8923821824793999
